In [ ]:
# ======Install Required Packages ======
!pip install --quiet transformers datasets faiss-cpu accelerate scikit-learn pandas tqdm openpyxl sentencepiece nlpaug torch

In [ ]:
# ====== Imports, Device Setup, and Focal Loss ======
import os
os.environ["WANDB_DISABLED"] = "true"  # Disable wandb if installed

import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import classification_report, precision_recall_fscore_support, accuracy_score
import faiss
import nlpaug.augmenter.word as naw
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DPRContextEncoder,
    DPRContextEncoderTokenizer,
    DPRQuestionEncoder,
    DPRQuestionEncoderTokenizer,
    MarianMTModel,
    MarianTokenizer
)

device = "cuda" if torch.cuda.is_available() else "cpu"

def focal_loss(logits: torch.Tensor, labels: torch.Tensor, alpha: float = 0.25, gamma: float = 2.0) -> torch.Tensor:
    """
    Compute Focal Loss for binary classification.
    logits: Tensor of shape [batch_size, 2]
    labels: Tensor of shape [batch_size] with values {0,1}
    alpha: weight for the minority class
    gamma: focusing parameter
    """
    ce = F.cross_entropy(logits, labels, reduction="none")
    pt = torch.exp(-ce)
    focal = alpha * (1 - pt) ** gamma * ce
    return focal.mean()

In [ ]:
# ====== Build Knowledge Corpus (DDI + EML) and Save to Excel ======
# Load DDI interactions, create text field
df_ddi = pd.read_csv("/content/DDI_data (3).csv")  # Update the path if needed
df_ddi = df_ddi[["drug1_name", "drug2_name", "interaction_type"]].dropna()
df_ddi["source"] = "DDI"
df_ddi["text"] = (
    df_ddi["drug1_name"] + " may interact with " + df_ddi["drug2_name"] + ": " + df_ddi["interaction_type"]
)
df_ddi.rename(
    columns={"drug1_name": "drug1", "drug2_name": "drug2", "interaction_type": "interaction"},
    inplace=True
)

# Load EML medical uses, create text field
df_eml = pd.read_excel("/content/EML export (2).xlsx")  # Update the path if needed
df_eml.columns = df_eml.columns.str.strip()
df_eml = df_eml[["Medicine name", "Indication"]].dropna()
df_eml["source"] = "EML"
df_eml["text"] = df_eml["Medicine name"] + " is used for: " + df_eml["Indication"]
df_eml.rename(columns={"Medicine name": "medicine_name", "Indication": "indication"}, inplace=True)

# Combine DDI and EML, drop duplicate texts
knowledge_df = pd.concat([df_ddi, df_eml], ignore_index=True)
knowledge_df.drop_duplicates(subset=["text"], inplace=True)

# Save the combined corpus to Excel
knowledge_df.to_excel("knowledge_corpus.xlsx", index=False)
print("Saved knowledge_corpus.xlsx with combined DDI+EML passages.")

Saved knowledge_corpus.xlsx with combined DDI+EML passages.


In [ ]:
# ====== Build FAISS Index Using DPR Context Encoder ======
# Load the knowledge corpus from Excel
knowledge_df = pd.read_excel("knowledge_corpus.xlsx")
passages = knowledge_df["text"].tolist()

# Create a Hugging Face Dataset for passages
passages_ds = Dataset.from_dict({
    "title": [f"doc_{i}" for i in range(len(passages))],
    "text": passages
})

# Compute DPR-context embeddings for each passage
ctx_tokenizer = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
ctx_encoder  = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base").to(device)

def embed_passages(batch):
    tokens = ctx_tokenizer(batch["text"], truncation=True, padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        output = ctx_encoder(**tokens)
    return {"embeddings": output.pooler_output.cpu().numpy()}

passages_ds = passages_ds.map(embed_passages, batched=True, batch_size=32, load_from_cache_file=False)

# Stack embeddings into a NumPy array and build FAISS index
embeddings = np.vstack(passages_ds["embeddings"]).astype("float32")
faiss.normalize_L2(embeddings)
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

# Save FAISS index and dataset to disk
faiss.write_index(index, "hf_knowledge_index.faiss")
passages_ds.save_to_disk("hf_knowledge_dataset")
print("Built and saved FAISS index: hf_knowledge_index.faiss")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DPRQuestionEncoderTokenizer'. 
The class this function is called from is 'DPRContextEncoderTokenizer'.
Some weights of the model checkpoint at facebook/dpr-ctx_encoder-single-nq-base were not used when initializing DPRContextEncoder: ['ctx_encoder.bert_model.pooler.dense

Map:   0%|          | 0/224170 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Saving the dataset (0/2 shards):   0%|          | 0/224170 [00:00<?, ? examples/s]

Built and saved FAISS index: hf_knowledge_index.faiss


In [ ]:
# ====== Load DPR Question Encoder and Define Retrieval Function ======
# Load DPR-question encoder
q_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
q_encoder   = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base").to(device)

# Define a function to retrieve top-k passages given a question
def retrieve_topk_passages(question: str, k: int = 3):
    tokens = q_tokenizer(question, truncation=True, padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        output = q_encoder(**tokens)
    q_embedding = output.pooler_output.cpu().numpy()
    faiss.normalize_L2(q_embedding)
    _, indices = index.search(q_embedding, k)
    return [passages[i] for i in indices[0].tolist()]

Some weights of the model checkpoint at facebook/dpr-question_encoder-single-nq-base were not used when initializing DPRQuestionEncoder: ['question_encoder.bert_model.pooler.dense.bias', 'question_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRQuestionEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRQuestionEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
# ====== Load QA Dataset, Perform 80/20 Split, Create combined_text, and Save to Excel ======
# Load QA data (questions + risk levels)
QA_DATA_PATH = "/content/MedInfo2019-QA-MedicationsFINAL (1) (4).xlsx"
df_qa = pd.read_excel(QA_DATA_PATH)
df_qa = df_qa.rename(columns={"Question": "question", "Risk_Level": "label"})

# Map "General"/"Critical" to 0/1
label_map = {"General": 0, "Critical": 1}
if df_qa["label"].dtype == object:
    df_qa["label_id"] = df_qa["label"].map(label_map)
else:
    df_qa["label_id"] = df_qa["label"]

# Stratified 80/20 train/validation split
train_df, val_df = train_test_split(
    df_qa,
    test_size=0.20,
    stratify=df_qa["label_id"],
    random_state=42
)

# Define a function that combines a question with top-k retrieved passages
def combine_question_with_passages(question: str, label: int, k: int = 3):
    topk = retrieve_topk_passages(question, k=k)
    combined_text = question + " [SEP] " + " [SEP] ".join(topk)
    return combined_text, label

# Build new lists for train and validation with combined_text
train_rows = []
for _, row in train_df.iterrows():
    combined_text, lbl = combine_question_with_passages(row["question"], row["label_id"], k=3)
    train_rows.append({"combined_text": combined_text, "label": lbl})

val_rows = []
for _, row in val_df.iterrows():
    combined_text, lbl = combine_question_with_passages(row["question"], row["label_id"], k=3)
    val_rows.append({"combined_text": combined_text, "label": lbl})

train_combined = pd.DataFrame(train_rows)
val_combined   = pd.DataFrame(val_rows)

# Save to Excel for later augmentation and cross-validation
train_combined.to_excel("train_rag_prepared.xlsx", index=False)
val_combined.to_excel("val_rag_prepared.xlsx", index=False)
print("Saved train_rag_prepared.xlsx and val_rag_prepared.xlsx with combined_text.")

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Saved train_rag_prepared.xlsx and val_rag_prepared.xlsx with combined_text.


In [ ]:
# ====== Augmentation via Back-Translation and Oversampling ======
# Load the prepared training data
df_orig = pd.read_excel("train_rag_prepared.xlsx")

# Load MarianMT models for English to French and French → English
model_name_en_fr = "Helsinki-NLP/opus-mt-en-fr"
model_name_fr_en = "Helsinki-NLP/opus-mt-fr-en"

tokenizer_en_fr = MarianTokenizer.from_pretrained(model_name_en_fr)
model_en_fr     = MarianMTModel.from_pretrained(model_name_en_fr).to(device)

tokenizer_fr_en = MarianTokenizer.from_pretrained(model_name_fr_en)
model_fr_en     = MarianMTModel.from_pretrained(model_name_fr_en).to(device)

def back_translate(text: str,
                   tokenizer_src: MarianTokenizer, model_src: MarianMTModel,
                   tokenizer_tgt: MarianTokenizer, model_tgt: MarianMTModel) -> str:
    """
    Perform back-translation: English to French to English.
    Returns a paraphrased English string.
    """
    device_local = "cuda" if torch.cuda.is_available() else "cpu"
    batch1 = tokenizer_src(text, return_tensors="pt", padding=True, truncation=True).to(device_local)
    trans1 = model_src.generate(**batch1)
    fr_text = tokenizer_src.batch_decode(trans1, skip_special_tokens=True)[0]
    batch2 = tokenizer_tgt(fr_text, return_tensors="pt", padding=True, truncation=True).to(device_local)
    trans2 = model_tgt.generate(**batch2)
    en_back = tokenizer_tgt.batch_decode(trans2, skip_special_tokens=True)[0]
    return en_back

# Filter only "Critical" examples (label=1) for augmentation
df_critical = df_orig[df_orig["label"] == 1].reset_index(drop=True)

augmented_rows = []
print("Starting back-translation augmentation for", len(df_critical), "critical examples …")
for idx, row in tqdm(df_critical.iterrows(), total=len(df_critical)):
    orig_text = row["combined_text"]
    try:
        bt = back_translate(orig_text, tokenizer_en_fr, model_en_fr, tokenizer_fr_en, model_fr_en)
    except:
        bt = orig_text
    augmented_rows.append({"combined_text": bt, "label": row["label"]})

# Concatenate original training data with augmented critical examples
df_augmented = pd.concat([df_orig, pd.DataFrame(augmented_rows)], ignore_index=True)
df_augmented = df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

# Display class distribution before and after augmentation
print("\n--- Class distribution before augmentation ---")
print(df_orig["label"].value_counts().rename({0: "General", 1: "Critical"}))
print("\n--- Class distribution after augmentation ---")
print(df_augmented["label"].value_counts().rename({0: "General", 1: "Critical"}))

/usr/local/lib/python3.11/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Starting back-translation augmentation for 90 critical examples …


100%|██████████| 90/90 [01:56<00:00,  1.30s/it]


--- Class distribution before augmentation ---
label
General     434
Critical     90
Name: count, dtype: int64

--- Class distribution after augmentation ---
label
General     434
Critical    180
Name: count, dtype: int64


In [ ]:
# ====== Stratified K-Fold Cross-Validation with BioBERT + Focal Loss ======
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_reports = []
f1_critical_scores = []

# Load BioBERT tokenizer
BIOBERT_CHECKPOINT = "dmis-lab/biobert-base-cased-v1.1"
hf_tokenizer = AutoTokenizer.from_pretrained(BIOBERT_CHECKPOINT)

for fold_idx, (train_idx, eval_idx) in enumerate(skf.split(df_augmented, df_augmented["label"]), start=1):
    print(f"\n\n========== Fold {fold_idx} / 5 ==========")

    # Split the DataFrame for this fold
    df_train_fold = df_augmented.iloc[train_idx].reset_index(drop=True)
    df_eval_fold  = df_augmented.iloc[eval_idx].reset_index(drop=True)

    # Ensure labels are int64
    df_train_fold["label"] = df_train_fold["label"].astype(np.int64)
    df_eval_fold["label"]  = df_eval_fold["label"].astype(np.int64)

    # Create Hugging Face Dataset directly from lists
    train_ds = Dataset.from_dict({
        "combined_text": df_train_fold["combined_text"].tolist(),
        "labels":        df_train_fold["label"].tolist()
    })
    eval_ds  = Dataset.from_dict({
        "combined_text": df_eval_fold["combined_text"].tolist(),
        "labels":        df_eval_fold["label"].tolist()
    })

    # Tokenization function
    def tokenize_fn(examples):
        return hf_tokenizer(
            examples["combined_text"],
            padding="max_length",
            truncation=True,
            max_length=384
        )

    tokenized_train = train_ds.map(tokenize_fn, batched=True, remove_columns=["combined_text"])
    tokenized_eval  = eval_ds.map(tokenize_fn, batched=True,   remove_columns=["combined_text"])

    # Keep only input_ids, attention_mask, labels
    cols_to_keep = ["input_ids", "attention_mask", "labels"]
    tokenized_train = tokenized_train.remove_columns([c for c in tokenized_train.column_names if c not in cols_to_keep])
    tokenized_eval  = tokenized_eval.remove_columns([c for c in tokenized_eval.column_names  if c not in cols_to_keep])

    # Set format to torch tensors
    tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    tokenized_eval.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

    # Calculate class weights based on training fold
    labels_train_fold = df_train_fold["label"].tolist()
    n_total = len(labels_train_fold)
    n_gen   = sum(1 for l in labels_train_fold if l == 0)
    n_cr    = sum(1 for l in labels_train_fold if l == 1)
    weight_gen = n_total / (2 * n_gen) if n_gen > 0 else 1.0
    weight_cr  = n_total / (2 * n_cr) if n_cr > 0 else 1.0
    class_weights = torch.tensor([weight_gen, weight_cr], dtype=torch.float32).to(device)

    # Load BioBERT for sequence classification (2 labels)
    model = AutoModelForSequenceClassification.from_pretrained(
        BIOBERT_CHECKPOINT,
        num_labels=2
    ).to(device)

    # Define metrics computation
    def compute_metrics(p):
        preds  = p.predictions.argmax(-1)
        labels = p.label_ids
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, preds, average=None, labels=[0,1]
        )
        acc = accuracy_score(labels, preds)
        return {
            "accuracy": float(acc),
            "precision_general": float(precision[0]),
            "recall_general":    float(recall[0]),
            "f1_general":        float(f1[0]),
            "precision_critical":float(precision[1]),
            "recall_critical":   float(recall[1]),
            "f1_critical":       float(f1[1])
        }

    # Create FocalTrainer subclass of Trainer
    class FocalTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.get("labels").to(device)
            outputs = model(
                input_ids=inputs.get("input_ids").to(device),
                attention_mask=inputs.get("attention_mask").to(device),
                labels=None
            )
            logits = outputs.logits
            loss = focal_loss(logits, labels, alpha=0.25, gamma=2.0)
            return (loss, outputs) if return_outputs else loss

    # Set up TrainingArguments for this fold
    training_args = TrainingArguments(
        output_dir=f"./results_fold{fold_idx}",
        num_train_epochs=8,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=2e-5,
        weight_decay=0.01,
        logging_dir=f"./logs_fold{fold_idx}",
        logging_steps=50,
        save_steps=200,
        save_total_limit=1,
        report_to="none"
    )

    # Instantiate FocalTrainer
    trainer = FocalTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_eval,
        tokenizer=hf_tokenizer,
        compute_metrics=compute_metrics
    )

    # Train on this fold
    print(f"\n Training Fold {fold_idx} …")
    trainer.train()

    # 8.15. Evaluate on validation set for this fold
    print(f"\n--- Evaluating Fold {fold_idx} ---")
    preds_output = trainer.predict(tokenized_eval)
    logits = preds_output.predictions
    y_pred = np.argmax(logits, axis=1)
    y_true = preds_output.label_ids

    rpt = classification_report(y_true, y_pred, target_names=["General", "Critical"], output_dict=True)
    all_reports.append(rpt)
    f1_critical_scores.append(rpt["Critical"]["f1-score"])

    print(classification_report(y_true, y_pred, target_names=["General","Critical"]))

# ================================================================#
# Summarize Cross-Validation Results
print("\n\n Cross-Validation Summary:")
for i, rpt in enumerate(all_reports, 1):
    f1c = rpt["Critical"]["f1-score"]
    print(f"  Fold {i}: F1 (Critical) = {f1c:.3f}")

avg_f1_crit = np.mean(f1_critical_scores)
print(f"\n   Average F1 (Critical) over 5 folds = {avg_f1_crit:.3f}")



========== Fold 1 / 5 ==========


Map:   0%|          | 0/491 [00:00<?, ? examples/s]

Map:   0%|          | 0/123 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-9-c358fc1ea329>:115: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `FocalTrainer.__init__`. Use `processing_class` instead.
  trainer = FocalTrainer(



 Training Fold 1 …


...

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-9-c358fc1ea329>:115: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `FocalTrainer.__init__`. Use `processing_class` instead.
  trainer = FocalTrainer(



 Training Fold 5 …


Step,Training Loss
50,0.040900
100,0.034500
150,0.023400
200,0.015200
250,0.006600
300,0.001600
350,0.002500
400,0.001000
450,0.000500



--- Evaluating Fold 5 ---


              precision    recall  f1-score   support

     General       0.95      0.93      0.94        86
    Critical       0.84      0.89      0.86        36

    accuracy                           0.92       122
   macro avg       0.90      0.91      0.90       122
weighted avg       0.92      0.92      0.92       122



 Cross-Validation Summary:
  Fold 1: F1 (Critical) = 0.722
  Fold 2: F1 (Critical) = 0.861
  Fold 3: F1 (Critical) = 0.827
  Fold 4: F1 (Critical) = 0.720
  Fold 5: F1 (Critical) = 0.865

   Average F1 (Critical) over 5 folds = 0.799


In [ ]:
# After you have a trained Trainer (e.g., `trainer`) and a validation dataset (`tokenized_eval`),
# you can print overall performance metrics as follows:

# 1. Evaluate on the validation set and print the built-in metrics (loss, accuracy, etc.)
eval_results = trainer.evaluate(tokenized_eval)
print("=== Trainer.evaluate() results ===")
for key, value in eval_results.items():
    print(f"{key}: {value}")

# 2. If you want a full classification report (precision/recall/F1 per class), run predictions:
import numpy as np
from sklearn.metrics import classification_report

# Run predictions on the validation set
pred_output = trainer.predict(tokenized_eval)
logits = pred_output.predictions             # shape: (num_examples, num_labels)
y_pred = np.argmax(logits, axis=1)
y_true = pred_output.label_ids               # ground‐truth labels

# Print a scikit-learn classification report
print("\n=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=["General", "Critical"]))

# 3. If you just want to print overall (macro/micro) metrics manually:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average="weighted")
rec = recall_score(y_true, y_pred, average="weighted")
f1 = f1_score(y_true, y_pred, average="weighted")

print("\n=== Overall (weighted) Metrics ===")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-score:  {f1:.4f}")

=== Trainer.evaluate() results ===
eval_loss: 0.018765101209282875
eval_accuracy: 0.9180327868852459
eval_precision_general: 0.9523809523809523
eval_recall_general: 0.9302325581395349
eval_f1_general: 0.9411764705882353
eval_precision_critical: 0.8421052631578947
eval_recall_critical: 0.8888888888888888
eval_f1_critical: 0.8648648648648649
eval_runtime: 1.0205
eval_samples_per_second: 119.549
eval_steps_per_second: 15.679
epoch: 8.0

=== Classification Report ===
              precision    recall  f1-score   support

     General       0.95      0.93      0.94        86
    Critical       0.84      0.89      0.86        36

    accuracy                           0.92       122
   macro avg       0.90      0.91      0.90       122
weighted avg       0.92      0.92      0.92       122


=== Overall (weighted) Metrics ===
Accuracy:  0.9180
Precision: 0.9198
Recall:    0.9180
F1-score:  0.9187
